In [ ]:
%pip install -q gensim
%pip install -q rank-bm25
%pip install -q sentence-transformers

In [ ]:
import pandas as pd
pd.options.display.float_format = "{:.4f}".format

In [ ]:
documents = [
    "I bought a car.",
    "I purchased an automobile.",
    "A dog bites a man.",
    "A man bites a dog."
]

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def get_cosine_similarity(encoded_vector):
    similarity = cosine_similarity(encoded_vector)
    return pd.DataFrame(similarity)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer() # Bag of Words
vectorized = vectorizer.fit_transform(documents)
df = pd.DataFrame(
    vectorized.toarray(),
    columns=vectorizer.get_feature_names_out()
)
print(df)

In [ ]:
print(get_cosine_similarity(vectorized))

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer() # TF-IDF
vectorized = vectorizer.fit_transform(documents)
df = pd.DataFrame(
    vectorized.toarray(),
    columns=vectorizer.get_feature_names_out()
)
print(df)

In [ ]:
print(get_cosine_similarity(vectorized))

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    model_name_or_path="all-MiniLM-L6-v2",
)
embeddings = model.encode(documents, convert_to_tensor=True)
print(embeddings.shape)
print(embeddings)

In [ ]:
print(get_cosine_similarity(embeddings))

In [ ]:
documents = [
    "커피를 마시면 잠이 잘 오지 않는다.",
    "카페인은 수면의 질에 영향을 줄 수 있다.",
    "햇빛은 우리 몸의 생체 리듬을 조절한다."
]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
embeddings = model.encode(documents, convert_to_tensor=True)
print(embeddings.shape)
print(embeddings)

In [ ]:
df = get_cosine_similarity(embeddings)
df.index = [s[:3] + "..." for s in documents]
df.columns = df.index
print(df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.figure(figsize=(6, 4))
sns.heatmap(df, annot=True, cmap='Blues', fmt='.2f', vmin=0, vmax=1, linewidths=0.5)
plt.title('Cosine Similarity Matrix', fontsize=15)
plt.show()

In [ ]:
documents = [
    "KDT는 K-Digital Training의 약자이다.",
    "KDT 14기는 경일대학교에서 수업을 진행한다.",
    "경북대학교는 KDT 교육과정을 운영한다.",
    "주니온 교수는 전이 학습, 웹, 클라우드 과목을 담당한다.",
    "KDT 14기는 아진산업 채용 예정자를 위한 교육과정이다.",
    "교육과정은 파이썬, 데이터 분석, 머신 러닝, 인공 지능을 포함한다.",
]

query = "주니온 교수가 인공 지능을 강의하는 장소는?"

In [ ]:
import pandas as pd
from rank_bm25 import BM25Okapi

bm25_corpus = [s.split() for s in documents]
bm25_query = query.split()

for i, doc in enumerate(bm25_corpus):
    print(f"Doc.{i}: {doc}")
print(f"Query: {bm25_query}")

In [ ]:
bm25 = BM25Okapi(bm25_corpus)
bm25_scores = bm25.get_scores(bm25_query)
df = pd.DataFrame({
    "document": documents,
    "score": bm25_scores
}).sort_values("score", ascending=False)
df

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
doc_embeddings = model.encode(documents, convert_to_tensor=True)
query_embedding = model.encode([query], convert_to_tensor=True)
print(doc_embeddings.shape)
print(query_embedding.shape)
print(query_embedding[0][:5])

In [ ]:
embedding_scores = cosine_similarity(query_embedding, doc_embeddings)[0]

df = pd.DataFrame({
    "document": documents,
    "similarity": embedding_scores
}).sort_values("similarity", ascending=False)
df